# Logic-Zero · Task 7b: Re-evaluate SFT checkpoint

Standalone re-evaluation of the saved best SFT checkpoint with **fixed parameters**:
- `max_new_tokens=1000` (was 700 — truncated 17-27% of n=4-6 generations)
- Per-n accuracy breakdown
- Frequent progress prints (every 5 puzzles)
- Strict GPU asserts (no silent CPU fallback)
- Auto-restore checkpoint from Drive

**Runtime:** GPU (T4 or L4). **Duration:** ~15-25 min.

**Before running:** Runtime → Restart runtime (clears any GPU memory from prior training).

## 1. Verify GPU + clean state

In [ ]:
import torch, gc
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    free, total = torch.cuda.mem_get_info()
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {free/1e9:.1f}GB free / {total/1e9:.1f}GB total')
    assert free / 1e9 > 5, 'Less than 5GB free — restart runtime first (Runtime → Restart runtime)'
else:
    raise RuntimeError('NO GPU — switch runtime type to GPU')
print(f'torch {torch.__version__}  bf16: {torch.cuda.is_bf16_supported()}')

## 2. Pull latest repo code

In [ ]:
import os
if not os.path.isdir('/content/logic-zero'):
    !git clone https://github.com/bsdnn/logic-zero.git /content/logic-zero
%cd /content/logic-zero
!git pull origin main

## 3. Install dependencies

Same as 01_sft_train — keeps Colab's torch + adds only what we need.

In [ ]:
import subprocess
def pip_inst(specs, retries=3):
    for i in range(retries):
        r = subprocess.run(['pip', 'install', '-q', '--default-timeout=180'] + specs,
                           capture_output=True, text=True)
        if r.returncode == 0:
            print(f'✓ {specs}')
            return
        print(f'[retry {i+1}/{retries}] {r.stderr[-300:]}')
    raise RuntimeError(f'pip install failed: {specs}')
pip_inst(['trl>=0.13.0,<0.15.0', 'peft>=0.13.0,<0.16.0', 'datasets>=3.0.0,<4.0.0', 'accelerate>=1.0.0'])
pip_inst(['z3-solver', 'python-dotenv'])  # train.common imports z3 at module top
import transformers, peft, z3
print(f'transformers={transformers.__version__}  peft={peft.__version__}  z3={z3.get_version_string()}')

## 4. Mount Drive + restore checkpoint

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import shutil, json

drive_ckpt = '/content/drive/MyDrive/logic-zero/checkpoints/sft/best'
local_ckpt = '/content/logic-zero/results/checkpoints/sft/best'

# Restore from Drive if local missing
if not os.path.isdir(local_ckpt):
    assert os.path.isdir(drive_ckpt), f'NO CHECKPOINT in Drive: {drive_ckpt}'
    os.makedirs(os.path.dirname(local_ckpt), exist_ok=True)
    shutil.copytree(drive_ckpt, local_ckpt)
    print(f'✓ Restored from Drive → {local_ckpt}')
else:
    print(f'✓ Local checkpoint already present: {local_ckpt}')

# Verify integrity
adapter = f'{local_ckpt}/adapter_model.safetensors'
assert os.path.exists(adapter), f'MISSING: {adapter}'
size_mb = os.path.getsize(adapter) / 1e6
assert size_mb > 10, f'Adapter too small ({size_mb}MB) — likely corrupted'
meta = json.load(open(f'{local_ckpt}/dev_acc.json'))
print(f'\nCheckpoint: {size_mb:.1f}MB')
print(f'Original dev_acc (max_new_tokens=700): {meta["acc"]:.3f} at epoch {meta["epoch"]}')
print('Files:', os.listdir(local_ckpt))

## 5. Load model + LoRA adapter (with strict GPU check)

In [ ]:
import time
from peft import PeftModel
from train.common import load_base_model

print('[1/2] Loading base Qwen2.5-1.5B...', flush=True)
t0 = time.time()
model, tok = load_base_model()
# load_base_model() returns model on CPU by default — must move to GPU
# explicitly (SFTTrainer does this internally during training, but eval
# code has to do it itself).
model = model.to('cuda')
print(f'  loaded in {time.time()-t0:.0f}s', flush=True)

print('[2/2] Loading LoRA adapter...', flush=True)
t0 = time.time()
model = PeftModel.from_pretrained(model, local_ckpt)
model.eval()
model.config.use_cache = True  # KV cache for ~10x faster generation
print(f'  loaded in {time.time()-t0:.0f}s', flush=True)

# Strict device check — no silent CPU fallback
dev = next(model.parameters()).device
assert dev.type == 'cuda', f'MODEL ON {dev} — generation will be 100x slower. Restart runtime!'
print(f'\n✓ Model on {dev}')
print(f'  GPU mem allocated: {torch.cuda.memory_allocated()/1e9:.2f}GB')

## 6. Run dev eval with `max_new_tokens=1000`

170 puzzles. Progress every 5. ETA ~15-25 min on T4, ~10-15 min on L4.

In [ ]:
from train.common import to_chat, extract_answer

dev_data = [json.loads(l) for l in open('data/dev_data.jsonl')]
print(f'Dev set: {len(dev_data)} puzzles')
from collections import Counter
print(f'Per-n distribution: {dict(sorted(Counter(len(r["ground_truth"]) for r in dev_data).items()))}\n')

per_n_correct, per_n_total = {}, {}
wrong_examples = []  # save a few wrongs for inspection
t0 = time.time()

with torch.no_grad():
    for i, rec in enumerate(dev_data):
        n = len(rec['ground_truth'])
        per_n_total[n] = per_n_total.get(n, 0) + 1
        inputs = tok(to_chat(tok, rec['puzzle']), return_tensors='pt').to(model.device)
        out = model.generate(
            **inputs, max_new_tokens=1000, do_sample=False,
            pad_token_id=tok.eos_token_id,
            stop_strings=['</answer>'], tokenizer=tok,
            temperature=None, top_p=None, top_k=None,
        )
        resp = tok.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
        pred = extract_answer(resp, n=n)
        is_correct = pred == rec['ground_truth']
        if is_correct:
            per_n_correct[n] = per_n_correct.get(n, 0) + 1
        elif len(wrong_examples) < 5 and n in (5, 6):
            wrong_examples.append({'n': n, 'gt': rec['ground_truth'], 'pred': pred, 'resp_tail': resp[-300:]})
        if (i+1) % 5 == 0:
            tot = sum(per_n_correct.values())
            eta = (time.time()-t0) / (i+1) * (len(dev_data) - i - 1)
            print(f'  {i+1:3d}/{len(dev_data)}  n={n}  acc={tot/(i+1):.3f}  '
                  f'elapsed={time.time()-t0:.0f}s  eta={eta:.0f}s', flush=True)

print(f'\n=== Done in {time.time()-t0:.0f}s ===')

## 7. Results: per-n breakdown + diagnosis

In [ ]:
print('=== Per-n accuracy (max_new_tokens=1000) ===')
for n in sorted(per_n_total):
    c, t = per_n_correct.get(n, 0), per_n_total[n]
    bar = '█' * int(c/t * 30)
    print(f'  n={n}: {c:3d}/{t:3d} = {c/t:.1%}  {bar}')

total_c, total_t = sum(per_n_correct.values()), sum(per_n_total.values())
new_acc = total_c / total_t
print(f'\nNEW dev_acc = {total_c}/{total_t} = {new_acc:.1%}')
print(f'OLD dev_acc (max_new_tokens=700) = {meta["acc"]:.1%}')
print(f'Improvement from fixing truncation: {(new_acc - meta["acc"])*100:+.1f}pp')

# Sanity criteria (notebook 01_sft_train spec)
print('\n=== Decision ===')
if new_acc >= 0.50:
    print('✅ dev_acc ≥ 50%: pipeline healthy. Proceed to DPO.')
elif new_acc >= 0.25:
    print('✅ dev_acc ≥ 25%: proceed to DPO, expect +5-10pp gain.')
else:
    print('⚠️ dev_acc < 25%: borderline. Check n=2/3 — if those ≥40%, DPO can still help on easy puzzles.')

## 8. Save results + sample errors

In [ ]:
result = {
    'dev_acc': new_acc,
    'old_dev_acc': meta['acc'],
    'best_epoch': meta['epoch'],
    'per_n': {str(n): {'correct': per_n_correct.get(n, 0), 'total': per_n_total[n],
                       'acc': per_n_correct.get(n, 0) / per_n_total[n]}
              for n in sorted(per_n_total)},
    'max_new_tokens': 1000,
    'wrong_samples_n5_6': wrong_examples,
}

os.makedirs('results', exist_ok=True)
with open('results/sft_dev_reeval.json', 'w') as f:
    json.dump(result, f, indent=2)
print('✓ Saved results/sft_dev_reeval.json')

# Backup to Drive
shutil.copy('results/sft_dev_reeval.json',
            '/content/drive/MyDrive/logic-zero/checkpoints/sft/sft_dev_reeval.json')
print('✓ Backed up to Drive')

# Inspect a few n=5/6 wrongs
if wrong_examples:
    print('\n=== Sample n=5/6 failures ===')
    for w in wrong_examples[:3]:
        print(f'\nn={w["n"]}  GT={w["gt"]}  PRED={w["pred"]}')
        print(f'  tail: ...{w["resp_tail"]}')

## ✅ Re-eval done

**Output:** `results/sft_dev_reeval.json` (local + Drive backup)

### Next steps based on result

| new dev_acc | Action |
|---|---|
| ≥ 25% | → Task 8: full eval on 1530-puzzle eval set, then Task 9: DPO |
| 15-25% | → Still proceed to DPO; expect gain mostly on n=2/3/4 |
| < 15% | → Inspect `wrong_samples_n5_6` for parse failures vs reasoning failures; may need to debug `extract_answer` regex |